In [ ]:
# !pip install ibm_boto3
# !pip uninstall kfp_components -y
# !pip install --no-cache-dir git+https://github.com/LukaszCmielowski/pipelines-components.git@rhoai_autorag_data_processing_pipeline

Import required libraries

In [6]:
import os
import json
import yaml
import urllib.request
from pathlib import Path

import ibm_boto3
from ibm_botocore.client import Config

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


Set bucket name and environment variables to connect to S3 instance
> **Note:** Bucket must already exists.

In [7]:
os.environ["AWS_ACCESS_KEY_ID"] = ""
os.environ["AWS_SECRET_ACCESS_KEY"] = ""
os.environ["AWS_S3_ENDPOINT"] = ""
os.environ["AWS_DEFAULT_REGION"] = ""
BUCKET_NAME = "autorag-dev-preview-dataset"

## Prepare experiment data

#### Initialize S3 client

In [8]:
s3_client = ibm_boto3.client(
    "s3",
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    config=Config(signature_version="s3v4"),
)

#### Upload documents
For the needs of this notebook we are using IBM financial reports available here: https://www.ibm.com/investor/financial-reporting

In [9]:
documents = [
    "https://www.ibm.com/downloads/documents/us-en/12bb2f913a3ba1a2",
    "https://www.ibm.com/downloads/documents/us-en/131cf8a39db327fd",
    "https://www.ibm.com/downloads/documents/us-en/131cf87ab633199f",
    "https://www.ibm.com/downloads/documents/us-en/1550f7eea8c0ded6",
    "https://www.ibm.com/downloads/documents/us-en/10a9980400afd114",
    "https://www.ibm.com/downloads/documents/us-en/10a9980468afdf4c",
    "https://www.ibm.com/downloads/documents/us-en/10a9980400afd11c",
    "https://www.ibm.com/downloads/documents/us-en/11ed3283ae56ec71"
]

for i, url in enumerate(documents):
    with urllib.request.urlopen(url) as response:
        content = response.read()
    s3_client.put_object(Bucket=BUCKET_NAME, Key=f"document_{i}.pdf", Body=content)

#### Upload benchmark dataset

In [10]:
benchmark = [
    {
        "question": "What was IBM's revenue in the first quarter of 2024?",
        "correct_answers": [
            "Revenue of $14.5 billion, up 1 percent, up 3 percent at constant currency."
        ],
        "correct_answer_document_ids": [
            "document_0.pdf"
        ]
    },
    {
        "question": "What did IBM announce regarding HashiCorp in first quarter 2024?",
        "correct_answers": [
            "IBM announced its intent to acquire HashiCorp, Inc. for $35 per share in cash, representing an enterprise value of $6.4 billion. The transaction was expected to close by the end of 2024."
        ],
        "correct_answer_document_ids": [
            "document_0.pdf"
        ]
    }
]

res = s3_client.put_object(Bucket=BUCKET_NAME, Key="benchmark.json", Body=json.dumps(benchmark))

Look up bucket contents

In [11]:
s3_client.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix="",
).get("Contents", [])

[{'Key': 'benchmark.json',
  'LastModified': datetime.datetime(2026, 2, 19, 18, 42, 2, 933000, tzinfo=tzutc()),
  'ETag': '"3d40f724eccdcc2febb1d3312f9ef593"',
  'ChecksumAlgorithm': ['CRC32'],
  'ChecksumType': 'FULL_OBJECT',
  'Size': 564,
  'StorageClass': 'STANDARD'},
 {'Key': 'document_0.pdf',
  'LastModified': datetime.datetime(2026, 2, 19, 18, 41, 53, 35000, tzinfo=tzutc()),
  'ETag': '"602598e51a6c8e06a7c80d26c4aee300"',
  'ChecksumAlgorithm': ['CRC32'],
  'ChecksumType': 'FULL_OBJECT',
  'Size': 230841,
  'StorageClass': 'STANDARD'},
 {'Key': 'document_1.pdf',
  'LastModified': datetime.datetime(2026, 2, 19, 18, 41, 54, 321000, tzinfo=tzutc()),
  'ETag': '"64c5beddfe7c50bb6b827d7f6d825118"',
  'ChecksumAlgorithm': ['CRC32'],
  'ChecksumType': 'FULL_OBJECT',
  'Size': 280313,
  'StorageClass': 'STANDARD'},
 {'Key': 'document_2.pdf',
  'LastModified': datetime.datetime(2026, 2, 19, 18, 41, 55, 494000, tzinfo=tzutc()),
  'ETag': '"9d0309206828a5c38e2e99aa036daa6d"',
  'ChecksumAl

## Process input documents
This step performs sampling and text extraction from the input documents
- Sampling is performed based on the benchmark dataset
- Docling library is used for text exrtraction

In [12]:
from kfp import local
from kfp_components.pipelines.data_processing.autorag.pipeline import data_processing_pipeline

local.init(
    runner=local.SubprocessRunner(use_venv=False),
    pipeline_root="./local_outputs",
    raise_on_error=True,
)

result = data_processing_pipeline(
    test_data_secret_name="autorag-input-data-secret",
    input_data_secret_name="autorag-input-data-secret",
    test_data_bucket_name="autorag-dev-preview-dataset",
    test_data_key="benchmark.json",
    input_data_bucket_name="autorag-dev-preview-dataset",
    input_data_key="",
    sampling_config={"max_size_gigabytes": 1},
)

19:42:31.965 - INFO - Running pipeline: 'autorag-data-processing-pipeline'
--------------------------------------------------------------------------------
19:42:31.966 - INFO - Executing task 'test-data-loader'
19:42:31.967 - INFO - Streamed logs:



/Users/wnowogor/PycharmProjects/ai4rag/.venv/lib/python3.14/site-packages/kfp/local/subprocess_task_handler.py:66: RuntimeWarning: You may be attemping to run a task that uses custom or non-Python base image 'wnowogorski-org/autorag_data_loading' in a Python environment. This may result in incorrect dependencies and/or incorrect behavior. Consider using the 'DockerRunner' to run this task in a container.
  warnings.warn(


    Looking in links: /var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/tmpnytfsgyu
    
    [notice] A new release of pip is available: 24.3.1 -> 26.0.1
    [notice] To update, run: /Users/wnowogor/PycharmProjects/ai4rag/.venv/bin/python -m pip install --upgrade pip
    [KFP Executor 2026-02-19 19:42:33,599 INFO]: Looking for component `test_data_loader` in --component_module_path `/var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/tmp.I2y5ttjFmu/ephemeral_component.py`
    [KFP Executor 2026-02-19 19:42:33,599 INFO]: Loading KFP component "test_data_loader" from /var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/tmp.I2y5ttjFmu/ephemeral_component.py (directory "/var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/tmp.I2y5ttjFmu" and module name "ephemeral_component")
    [KFP Executor 2026-02-19 19:42:33,600 INFO]: Got executor_input:
    {
        "inputs": {
            "parameterValues": {
                "test_data_bucket_name": "autorag-dev-preview-dataset",
                "test_data_pa

/Users/wnowogor/PycharmProjects/ai4rag/.venv/lib/python3.14/site-packages/kfp/local/subprocess_task_handler.py:66: RuntimeWarning: You may be attemping to run a task that uses custom or non-Python base image 'wnowogorski-org/autorag_data_loading' in a Python environment. This may result in incorrect dependencies and/or incorrect behavior. Consider using the 'DockerRunner' to run this task in a container.
  warnings.warn(


    Looking in links: /var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/tmpcofvbky2
    
    [notice] A new release of pip is available: 24.3.1 -> 26.0.1
    [notice] To update, run: /Users/wnowogor/PycharmProjects/ai4rag/.venv/bin/python -m pip install --upgrade pip
    [KFP Executor 2026-02-19 19:42:36,539 INFO]: Looking for component `documents_sampling` in --component_module_path `/var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/tmp.Z0UYfvfErt/ephemeral_component.py`
    [KFP Executor 2026-02-19 19:42:36,539 INFO]: Loading KFP component "documents_sampling" from /var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/tmp.Z0UYfvfErt/ephemeral_component.py (directory "/var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/tmp.Z0UYfvfErt" and module name "ephemeral_component")
    [KFP Executor 2026-02-19 19:42:36,540 INFO]: Got executor_input:
    {
        "inputs": {
            "artifacts": {
                "test_data": {
                    "artifacts": [
                        {
           

/Users/wnowogor/PycharmProjects/ai4rag/.venv/lib/python3.14/site-packages/kfp/local/subprocess_task_handler.py:66: RuntimeWarning: You may be attemping to run a task that uses custom or non-Python base image 'wnowogorski-org/autorag_data_loading' in a Python environment. This may result in incorrect dependencies and/or incorrect behavior. Consider using the 'DockerRunner' to run this task in a container.
  warnings.warn(


    Looking in links: /var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/tmpha8lvzlk
    
    [notice] A new release of pip is available: 24.3.1 -> 26.0.1
    [notice] To update, run: /Users/wnowogor/PycharmProjects/ai4rag/.venv/bin/python -m pip install --upgrade pip
    [KFP Executor 2026-02-19 19:42:39,215 INFO]: Looking for component `text_extraction` in --component_module_path `/var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/tmp.ttNwzgYdSv/ephemeral_component.py`
    [KFP Executor 2026-02-19 19:42:39,215 INFO]: Loading KFP component "text_extraction" from /var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/tmp.ttNwzgYdSv/ephemeral_component.py (directory "/var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/tmp.ttNwzgYdSv" and module name "ephemeral_component")
    [KFP Executor 2026-02-19 19:42:39,216 INFO]: Got executor_input:
    {
        "inputs": {
            "artifacts": {
                "sampled_documents_descriptor": {
                    "artifacts": [
                        

Look up sampling configuraton

In [13]:
pipeline_root = Path("./local_outputs")
run_dirs = [d for d in pipeline_root.iterdir() if d.is_dir()]
last_run = max(run_dirs, key=lambda d: d.stat().st_mtime)
path = next(last_run.rglob("sampled_documents_descriptor.yaml"), None)
if path is None:
    raise FileNotFoundError(f"sampled_documents_descriptor.yaml not under {last_run}")
with open(path) as f:
    descriptor = yaml.safe_load(f)
    
descriptor

{'bucket': 'autorag-dev-preview-dataset',
 'prefix': '',
 'documents': [{'key': 'document_1.pdf', 'size_bytes': 280313},
  {'key': 'document_0.pdf', 'size_bytes': 230841},
  {'key': 'document_2.pdf', 'size_bytes': 275585},
  {'key': 'document_3.pdf', 'size_bytes': 278516},
  {'key': 'document_4.pdf', 'size_bytes': 211288},
  {'key': 'document_5.pdf', 'size_bytes': 280378},
  {'key': 'document_6.pdf', 'size_bytes': 278765},
  {'key': 'document_7.pdf', 'size_bytes': 290324}],
 'total_size_bytes': 2126010,
 'count': 8}

Documents processing results

In [18]:
from pathlib import Path
from langchain_core.documents import Document

pipeline_root = Path("./local_outputs")
run_dirs = [d for d in pipeline_root.iterdir() if d.is_dir()]
last_run = max(run_dirs, key=lambda d: d.stat().st_mtime)
md_files = sorted(last_run.rglob("*.md"))

documents = [
    Document(
        page_content=p.read_text(encoding="utf-8", errors="replace"),
        metadata={"document_id": p.stem},
    )
    for p in md_files
]
documents[0]

Document(metadata={'document_id': 'document_0'}, page_content='## IBM RELEASES FIRST-QUARTER RESULTS\n\nResults exceed expectations driven by strong Software revenue growth, significant gross margin expansion and solid free cash flow\n\nARMONK, N.Y., April 23, 2025 . . . IBM (NYSE: IBM) today announced first-quarter 2025 earnings results.\n\n\'We exceeded expectations for revenue, profitability and free cash flow in the quarter, led by strength across our Software portfolio. There continues to be strong demand for generative AI and our book of business stands at more than $6 billion inception-to-date, up more than $1 billion in the quarter," said Arvind Krishna, IBM chairman, president and chief executive officer. "We remain bullish on the long-term growth opportunities for technology and the global economy. While the macroeconomic environment is fluid, based on what we know today, we are maintaining our full-year expectations for revenue growth and free cash flow."\n\n## First-Quarter

## Run ai4rag experiment